# Kamera autoencoder

CARLA kepek (`dataset/camera/`) -> autoencoder, MSE loss, latent 128.

In [ ]:
import gc
import glob
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

from camera.camera_ae import CameraAutoEncoder
from camera.resnet_ae import ResNetAE
from camera.vgg_ae import VGGAE

DATA_DIR = "dataset/camera"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
np.random.seed(0)
print(DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## Adatok

In [ ]:
def load_images(data_dir=DATA_DIR):
    """PNG-k -> (N, 3, 80, 160) float32 [0,1]."""
    paths = sorted(glob.glob(os.path.join(data_dir, "*.png")))
    out = np.empty((len(paths), 80, 160, 3), dtype=np.uint8)
    for i, p in enumerate(tqdm(paths, desc="betoltes", unit="kep")):
        out[i] = np.asarray(Image.open(p).convert("RGB"))
    return torch.from_numpy(out).permute(0, 3, 1, 2).float().div_(255.0)


t0 = time.time()
images = load_images()
print(f"{tuple(images.shape)}  {images.numel() * 4 / 1e9:.1f} GB  "
      f"({time.time() - t0:.0f} s)")
print(f"ertekek [{images.min():.2f}, {images.max():.2f}]  atlag {images.mean():.3f}")

In [ ]:
# Keveres. A kepek 20 Hz-cel, idorendben jottek - keveres nelkul egy batch
# ugyanannak a par masodpercnek a majdnem azonos kepeibol allna.
images = images[torch.randperm(len(images))]

n_val = int(len(images) * 0.15)
x_val, x_train = images[:n_val], images[n_val:]
print(f"train {len(x_train)}  val {len(x_val)}")

In [ ]:
BATCH = 128
train_loader = DataLoader(TensorDataset(x_train), batch_size=BATCH, shuffle=True)
val_loader = DataLoader(TensorDataset(x_val), batch_size=BATCH)
print(f"batch {BATCH}, train batch-ek {len(train_loader)}")

In [ ]:
def show(x, n=8, title=""):
    idx = np.random.choice(len(x), n, replace=False)
    fig, axes = plt.subplots(1, n, figsize=(2 * n, 2.2))
    for ax, i in zip(axes, idx):
        ax.imshow(x[i].permute(1, 2, 0).numpy())
        ax.axis("off")
    fig.suptitle(title)
    plt.show()


show(images, title="Minta kepek")

## Tanitas

In [ ]:
def evaluate(model, loader):
    model.eval()
    total = 0.0
    with torch.no_grad():
        for (xb,) in loader:
            xb = xb.to(DEVICE)
            total += F.mse_loss(model(xb), xb).item() * len(xb)
    return total / len(loader.dataset)


def train_model(model, epochs=30, lr=1e-3, patience=5, label=""):
    """MSE-vel tanit. A vegen a legjobb val loss-hoz tartozo sulyok maradnak."""
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    hist = {"label": label, "train": [], "val": [], "best": float("inf"),
            "params": sum(p.numel() for p in model.parameters())}
    best_state, bad = None, 0

    print(f"{label}  ({hist['params'] / 1e6:.1f}M parameter)")
    for ep in range(1, epochs + 1):
        model.train()
        run = 0.0
        bar = tqdm(train_loader, desc=f"epoch {ep}/{epochs}", leave=False)
        for (xb,) in bar:
            xb = xb.to(DEVICE)
            loss = F.mse_loss(model(xb), xb)
            opt.zero_grad()
            loss.backward()
            opt.step()
            run += loss.item() * len(xb)
            bar.set_postfix(loss=f"{loss.item():.5f}")

        tr = run / len(train_loader.dataset)
        va = evaluate(model, val_loader)
        hist["train"].append(tr)
        hist["val"].append(va)

        if va < hist["best"]:
            hist["best"], bad = va, 0
            best_state = {k: v.detach().cpu().clone()
                          for k, v in model.state_dict().items()}
            mark = " *"
        else:
            bad += 1
            mark = ""
        print(f"  {ep:2d}  train {tr:.5f}  val {va:.5f}{mark}")

        if bad >= patience:
            print("  early stop")
            break

    model.load_state_dict(best_state)

    # Mentes sajat nevvel. A modell osztalyanak neve is bekerul, hogy
    # visszatoltesnel tudd, melyik architektura.
    ckpt = f"camera/{label}.ckpt"
    torch.save({"state_dict": model.state_dict(),
                "hparams": dict(model.hparams),
                "arch": type(model).__name__,
                "best_val": hist["best"],
                "train": hist["train"], "val": hist["val"]}, ckpt)
    print(f"  mentve: {ckpt}  (best val {hist['best']:.5f})")

    # A modellt CPU-ra tesszuk, es kiuritjuk a GPU cache-t. Harom modellt
    # tanitunk egymas utan, es mindegyik a memoriaban maradna - egy 8 GB-os
    # kartya ettol elfogy. A rekonstrukciok rajzolasakor a show_reconstructions
    # ideiglenesen visszateszi a GPU-ra.
    model.cpu()
    hist["model"] = model
    gc.collect()
    torch.cuda.empty_cache()
    return hist

In [ ]:
histories = []

model = CameraAutoEncoder(latent_dim=256)
histories.append(train_model(model, epochs=30, label="camera_ae"))

## Eredmenyek

In [ ]:
def plot_curves(histories):
    plt.figure(figsize=(8, 4.5))
    colors = plt.cm.tab10(np.linspace(0, 1, 10))
    for k, h in enumerate(histories):
        ep = range(1, len(h["train"]) + 1)
        plt.plot(ep, h["train"], "-", color=colors[k], label=f"{h['label']} train")
        plt.plot(ep, h["val"], "--", color=colors[k], label=f"{h['label']} val")
    plt.xlabel("epoch")
    plt.ylabel("MSE")
    plt.yscale("log")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.title("Tanulasi gorbek")
    plt.show()

    for h in sorted(histories, key=lambda d: d["best"]):
        print(f"{h['label']:16s} best val {h['best']:.5f}  "
              f"({h['params'] / 1e6:.1f}M, {len(h['train'])} epoch)")


plot_curves(histories)

In [ ]:
def show_reconstructions(histories, x, n=8):
    """Felul az eredeti, alatta soronkent egy-egy modell kimenete."""
    idx = np.random.choice(len(x), n, replace=False)
    orig = x[idx].to(DEVICE)

    rows = len(histories) + 1
    fig, axes = plt.subplots(rows, n, figsize=(2.1 * n, 2.1 * rows))
    axes = np.atleast_2d(axes)
    for j in range(n):
        axes[0, j].imshow(orig[j].cpu().permute(1, 2, 0).numpy())
        axes[0, j].axis("off")
    axes[0, 0].set_title("eredeti", loc="left", fontsize=9)

    for i, h in enumerate(histories, start=1):
        m = h["model"].to(DEVICE).eval()
        with torch.no_grad():
            rec = m(orig).cpu().clamp(0, 1)
        m.cpu()                      # ne maradjon bent a GPU-n
        for j in range(n):
            axes[i, j].imshow(rec[j].permute(1, 2, 0).numpy())
            axes[i, j].axis("off")
        axes[i, 0].set_title(h["label"], loc="left", fontsize=9)

    plt.tight_layout()
    plt.show()


show_reconstructions(histories, x_val)

In [ ]:
# Minden modell a sajat nevevel mentodott tanitas utan:
for f in sorted(glob.glob("camera/*.ckpt")):
    d = torch.load(f, map_location="cpu", weights_only=False)
    print(f"{f:28s} {d['arch']:18s} best val {d['best_val']:.5f}")

## ResNet-stilusu architektura

Ugyanaz a latent (256) es ugyanaz a loss (MSE), csak mas a halo felepitese:

- **residual blokkok** (`y = x + F(x)`): a reteg csak a VALTOZAST tanulja, nem
  a teljes kimenetet, es a gradiens a skip agon akadalytalanul visszafolyik
- **GroupNorm** BatchNorm helyett: nem fugg a batch merettol
- **Upsample + Conv** ConvTranspose helyett: nincs sakktabla-minta
  (checkerboard artifact)

A skip connection a BLOKKON BELUL van - az encoderbol nem megy at semmi
kozvetlenul a decoderbe, tehat minden informacio a latenten keresztul halad.
(Ezert nem hasznalhato itt klasszikus U-Net: ott a skip megkerulne a
latentet, es az uresen maradna.)

In [ ]:
resnet = ResNetAE(latent_dim=256)
histories.append(train_model(resnet, epochs=30, label="resnet_ae"))

## VGG-stilusu architektura

A `jzenn/Image-AutoEncoder` halo: a VGG-19 elso negy blokkja encoderkent,
tukorkep dekoder. Ket sajatossaga:

- **ReflectionPad**: a konvoluciok elott tukrozott padding, nem nulla. A
  nulla-padding sotet keretet rajzolna a kep szelere.
- **Elotanitott sulyok**: az encoder ImageNet-en tanult VGG-19 sulyokkal
  indul. Enelkul a halo NEM tanul - 22 reteg normalizalas nelkul, a jel
  elhal, es a Sigmoid mindent 0.5-re visz.

**A tanulasi rata itt 1e-4**, nem 1e-3 mint a masik kettonel: 1e-3 mellett
a halo elszall (a loss felugrik es beragad).

In [ ]:
vgg = VGGAE(latent_dim=256)
histories.append(train_model(vgg, epochs=30, lr=1e-4, label="vgg_ae"))

In [ ]:
plot_curves(histories)
show_reconstructions(histories, x_val)

In [ ]:
def latent_stats(histories, x, n=1000):
    """Kihasznalja-e a halo a latent teret?

    halott dim : akinek a szorasa ~0, az nem hordoz informaciot
    telitett   : a tanh(+-1) hataran ulo ertekek - ezek gradiense majdnem
                 nulla, tehat gyakorlatilag nem tanulnak tovabb
    """
    idx = np.random.choice(len(x), min(n, len(x)), replace=False)
    fig, axes = plt.subplots(1, len(histories), figsize=(6 * len(histories), 3.2),
                             squeeze=False)

    for k, h in enumerate(histories):
        m = h["model"].to(DEVICE).eval()
        scale = m.hparams.latent_scale
        with torch.no_grad():
            z = m.encode(x[idx].to(DEVICE)).cpu().numpy()
        m.cpu()

        std = z.std(axis=0)
        dead = int((std < 0.01 * scale).sum())
        sat = 100 * (np.abs(z) > 0.97 * scale).mean()

        ax = axes[0, k]
        ax.bar(range(len(std)), np.sort(std)[::-1])
        ax.set_title(f"{h['label']}: {dead} halott / {len(std)}, "
                     f"{sat:.0f}% telitett")
        ax.set_xlabel("latent dimenzio")
        ax.set_ylabel("szoras")

        print(f"{h['label']:14s} halott {dead:3d}/{len(std):3d}  "
              f"telitett {sat:4.0f}%  best val {h['best']:.5f}")

    plt.tight_layout()
    plt.show()


latent_stats(histories, x_val)